# Train the model which predicts mortality risk

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

train_data_path = os.getenv('TRAINING_DATA_SECONDARY_OUTCOME')
train = pd.read_csv(train_data_path)
train = train.drop(columns=['Studio', 'ESRD', 'Creat1', 'esrd_upper', 'esrd_lower'])
train.shape

(3599, 17)

In [2]:
train.columns

Index(['Sesso', 'Diabete', 'Eta1', 'Prot1', 'Epi1', 'Colesterolo1', 'Hb1',
       'Ca1', 'P1', 'BMI1', 'Death_preHD', 'fu_morte_1', 'ckd_cause_hypertens',
       'ckd_cause_diabet', 'ckd_cause_glom_dis', 'ckd_cause_tubul_inter',
       'ckd_cause_pkd'],
      dtype='object')

In [3]:
features = ['Eta1', 'Epi1', 'Hb1', 
            'Prot1', 'Colesterolo1', 'P1', 
            'Ca1', 'Sesso', 'ckd_cause_glom_dis', 
            'Death_preHD', 'fu_morte_1']
train = train[features]

In [4]:
train.rename(columns={'Epi1': 'EGFR', 
                      'Prot1': 'URIN_PROT', 
                      'Eta1': 'AGE', 
                      'P1': 'PHOSPHORUS', 
                      'Ca1': 'CALCIUM', 
                      'Hb1': 'HB', 
                      'Colesterolo1': 'CHOLESTEROL',
                      'Sesso': 'MALE',
                      'ckd_cause_glom_dis': 'GLOM_DISEASE',
                      'fu_morte_1': 'time_to_mortality',
                      'Death_preHD': 'mortality'}, inplace=True)

In [5]:
from sksurv.ensemble import RandomSurvivalForest
import joblib
X = train.drop(columns=['time_to_mortality', 'mortality'])
y = np.array([(bool(event), time) for event, time in zip(train['mortality'], train['time_to_mortality'])],
             dtype=[('mortality', bool), ('time_to_mortality', float)])
rsf = RandomSurvivalForest(random_state=1,
                           max_depth=5,
                           max_features='sqrt',
                           min_samples_leaf=10,
                           min_samples_split=3,
                           n_estimators=100)
rsf.fit(X,y)
joblib.dump(rsf, 'rsf.joblib', compress=3)

['rsf.joblib']

In [6]:
import json, sys, sksurv
meta = {
    "python": sys.version,
    "numpy": np.__version__,
    "sksurv": sksurv.__version__,
}
json.dump(meta, open("rsf_metadata.json", "w"))